In [1]:

import torch, torchvision
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import DataLoader
import time
import os
import cv2
from dataPreparation import dataPrep
from imutils import paths
from sklearn.model_selection import train_test_split
from torch.nn import Module
from FCN import FCN
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from tqdm import tqdm
from earlyStopping import EarlyStopping


In [2]:

dataset_path = "/Users/beyzaecemerce/Desktop/GitHub/thesis/drone"
image_path=dataset_path+'/dataset/semantic_drone_dataset/original_images'
masked_path = dataset_path+ '/RGB_color_image_masks/RGB_color_image_masks'
TEST_SPLIT = 0.20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = True if DEVICE == "cuda" else False

In [18]:
image_path

'/Users/beyzaecemerce/Desktop/GitHub/thesis/drone/dataset/semantic_drone_dataset/original_images'

In [19]:
image = cv2.imread(image_path+"/100.jpg")
mask=cv2.imread(masked_path+"/100.png")
print(image.shape)
print(mask.shape)
figure, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 10))
ax[0].grid(False)
ax[1].grid(False)
ax[0].imshow(image)
ax[1].imshow(mask)
ax[0].set_title("Image")
ax[1].set_title("Original Mask")
figure.tight_layout()
figure.show()

(4000, 6000, 3)
(4000, 6000, 3)


/var/folders/xy/y85s_w552q1_8cd2gzzrqfmc0000gn/T/ipykernel_5914/754042599.py:13: UserWarning: Matplotlib is currently using module://matplotlib_inline.backend_inline, which is a non-GUI backend, so cannot show the figure.
  figure.show()


In [3]:
INIT_LR = 2e-4
NUM_EPOCHS = 300
BATCH_SIZE = 32

INPUT_IMAGE_WIDTH = 6000
INPUT_IMAGE_HEIGHT = 4000

BASE_OUTPUT = dataset_path+"/output"

MODEL_PATH = os.path.join(BASE_OUTPUT, "MODEL.pth")
PLOT_PATH = os.path.sep.join([BASE_OUTPUT, "Train_Test_Plot.png"])
TEST_PATHS = os.path.sep.join([BASE_OUTPUT, "test_path.txt"])

In [4]:
imagePaths = sorted(list(paths.list_images(image_path)))
maskPaths = sorted(list(paths.list_images(masked_path)))

split = train_test_split(imagePaths, maskPaths,test_size=TEST_SPLIT, random_state=42)

(trainImages, testImages) = split[:2]
(trainMasks, testMasks) = split[2:]

print("[INFO] saving testing image paths...")
f = open(TEST_PATHS, "w")
f.write("\n".join(testImages))
f.close()

[INFO] saving testing image paths...


In [5]:
input_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((384, 512),interpolation=cv2.INTER_NEAREST),  # Resize the input image to (height=256, width=384)
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
mask_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((384, 512)),  # Resize the mask to the same size as the input image
    transforms.ToTensor()
])

In [6]:
train = dataPrep(imagePaths=trainImages, maskPaths=trainMasks, input_transform=input_transform, mask_transform=mask_transform)
test = dataPrep(imagePaths=testImages, maskPaths=testMasks, input_transform=input_transform, mask_transform=mask_transform)


trainLoader = DataLoader(train, shuffle=False,batch_size=BATCH_SIZE, pin_memory=PIN_MEMORY,num_workers=os.cpu_count())

testLoader = DataLoader(test, shuffle=False,batch_size=BATCH_SIZE, pin_memory=PIN_MEMORY,num_workers=os.cpu_count())


In [ ]:
for (i, (x, y)) in enumerate(trainLoader):
    if i==1:
        (x, y) = (x.to(DEVICE), y.to(DEVICE))
        print(y)

tensor([[[[0.3373, 0.3412, 0.3412,  ..., 0.5020, 0.5020, 0.4863],
          [0.3294, 0.3412, 0.3412,  ..., 0.5020, 0.5020, 0.4863],
          [0.3294, 0.3412, 0.3412,  ..., 0.5020, 0.5020, 0.4863],
          ...,
          [0.3412, 0.3412, 0.3412,  ..., 0.5020, 0.5020, 0.4863],
          [0.3412, 0.3412, 0.3412,  ..., 0.5020, 0.5020, 0.4863],
          [0.3294, 0.3294, 0.3294,  ..., 0.4863, 0.4863, 0.4706]],

         [[0.4000, 0.4039, 0.4039,  ..., 0.2510, 0.2510, 0.2431],
          [0.3922, 0.4039, 0.4039,  ..., 0.2510, 0.2510, 0.2431],
          [0.3882, 0.4039, 0.4039,  ..., 0.2510, 0.2510, 0.2431],
          ...,
          [0.4039, 0.4039, 0.4039,  ..., 0.2510, 0.2510, 0.2431],
          [0.4039, 0.4039, 0.4039,  ..., 0.2510, 0.2510, 0.2431],
          [0.3922, 0.3922, 0.3922,  ..., 0.2431, 0.2431, 0.2353]],

         [[0.4353, 0.4392, 0.4392,  ..., 0.5020, 0.5020, 0.4863],
          [0.4275, 0.4392, 0.4392,  ..., 0.5020, 0.5020, 0.4863],
          [0.4235, 0.4392, 0.4392,  ..., 0

In [7]:
model=FCN(20)
opt = Adam(model.parameters(), lr=INIT_LR)
lossFunc=torch.nn.CrossEntropyLoss()
trainSteps = len(train) // BATCH_SIZE
testSteps = len(test) // BATCH_SIZE
H = {"train_loss": [], "test_loss": []}


In [8]:
print("[INFO] training the network...")
startTime = time.time()
for e in tqdm(range(NUM_EPOCHS)):
  model.train()
  totalTrainLoss = 0
  totalTestLoss = 0

  for (i, (x, y)) in enumerate(trainLoader):
    (x, y) = (x.to(DEVICE), y.to(DEVICE))

    print("x",x.shape)
    print("y",y.shape)
    pred = model(x)
    print(pred.shape)
    #pred = torch.argmax(pred, dim=1)
    print(y.shape)
    loss = lossFunc(pred, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    totalTrainLoss += loss

  with torch.no_grad():
    model.eval()
    for (x, y) in testLoader:
      (x, y) = (x.to(DEVICE), y.to(DEVICE))
      pred = model(x)
      totalTestLoss += lossFunc(pred, y)
  avgTrainLoss = totalTrainLoss / trainSteps
  avgTestLoss = totalTestLoss / testSteps
  H["train_loss"].append(avgTrainLoss.cpu().detach().numpy())
  H["test_loss"].append(avgTestLoss.cpu().detach().numpy())
  early_stopping(avgTrainLoss,avgTestLoss)
        
  if early_stopping.early_stop:
      print("Early stopping")
      break
  print("[INFO] EPOCH: {}/{}".format(e + 1, NUM_EPOCHS))
  print("Train loss: {:.6f}, Test loss: {:.4f}".format(avgTrainLoss, avgTestLoss))
endTime = time.time()
print("[INFO] total time taken to train the model: {:.2f}s".format(endTime - startTime))

[INFO] training the network...


  0%|          | 0/300 [00:00<?, ?it/s]

x torch.Size([32, 3, 384, 512])
y torch.Size([32, 3, 384, 512])
conv1 torch.Size([32, 64, 384, 512])
conv2 torch.Size([32, 64, 384, 512])
pool1_out torch.Size([32, 64, 192, 256])
pool2_out torch.Size([32, 128, 96, 128])
pool3_out torch.Size([32, 256, 48, 64])
pool4_out torch.Size([32, 512, 24, 32])
conv13 torch.Size([32, 512, 24, 32])
pool5_out torch.Size([32, 512, 12, 16])
bn1 torch.Size([32, 512, 24, 32])
bn2 torch.Size([32, 256, 48, 64])
bn3 torch.Size([32, 128, 96, 128])
bn4 torch.Size([32, 64, 192, 256])
bn5 torch.Size([32, 32, 384, 512])
torch.Size([32, 20, 384, 512])
torch.Size([32, 3, 384, 512])


  0%|          | 0/300 [01:05<?, ?it/s]


RuntimeError: only batches of spatial targets supported (3D tensors) but got targets of dimension: 4